# nano Beatrix v1 — the last block's big direction, screened before the train

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_lastfix_screen.ipynb)

**Why this runs first.** In the first minutes of the v1 train, under Beatrix 2s's recipe, every craft turned most of its last block's output onto one fixed direction, the form the larger 2s crafts and Beatrix 3 ended in. Read on 32 held-out documents:

| | at birth | v1, step 1,145 (300M bytes of wikitext) | the same body after its whole bed train (Adam, about 30,000 steps) | the 2s crafts |
|---|---|---|---|---|
| share of the last block's output on its one mean direction | 0.19 - 0.24 | **0.81 - 0.82** | 0.43 - 0.50 | 0.98 by step 20,219; 0.99 by 45,399 |
| the last bank's write: share on its own direction | 0.30 - 0.35 | 0.75 - 0.81 | 0.30 - 0.38 | |
| the last bank's write: size per byte | 10 | 250 - 373 | 75 - 91 | grows 2.7x over 25,000 steps, no ceiling |
| the last block's rank (of 256) | 62 - 68 | 13 - 16 | 37 - 44 | |

The library carries four bounds on that write (`bank_last_fix`), built after the 2s finding and never trained until now:

| form | what it does to the last block's write | at birth |
|---|---|---|
| `register` | the write's part along its slowly tracked mean direction becomes a declared number; the stream keeps it times one gain shared by every byte, which holds that part's average size near its size after the first 200 training passes (the gain clamped to 1/4 - 4); the head reads the number on its own small path | the plain craft exactly |
| `temperature` | that part leaves the stream; the head divides its logits by a per-byte temperature read from it | the plain craft exactly (it acts from the first training pass) |
| `gain` | the whole write times one gain shared by every byte, holding its average size near the first 200 passes' | the plain craft exactly |
| `normwrite` | the write standardized over the row's positions so far: the comparison arm, expected to cost bits (it removes the per-byte confidence that direction carries) | not the plain craft |

**The arms** (each from its v1 craft's own start, the same data in the same order, the 2s recipe, 4,960 steps: `warmup_wikitext` 0.3B then the first 1.0B of `fineweb_main`):

| arm | body | start | form |
|---|---|---|---|
| `c-bare-s3` | Castor | the v1 Castor start (seed 3) | none |
| `c-register` / `c-temperature` / `c-gain` | Castor | seed 3 | each form |
| `c-bare-s4` | Castor | seed 4 | none: the seed bar |
| `c-normwrite` | Castor | seed 3 | the comparison arm |
| `pdc-bare` / `pdc-register` / `pdc-temperature` | Pollux's layout, dual hubs + Canon | the v1 start (seed 1) | none / register / temperature |

**What is read** every 500 steps and at both phase ends (TensorBoard `lastwrite/*` beside the trainer's own curves, and each arm's record): bits a byte on the fineweb-edu holdout; the share on the last block's direction at every block and after the final norm (what the bound lets through); the last bank's raw write before the bound, its share and size (how hard the craft pushes against the bound); every block's rank; the final norm's gain and bias against the direction; the form's own gain. At the end: the drift memory on random bytes (the exit's H; 0.5 = none), the larger crafts' signature that a bound must not erase.

**How a form passes** (written before any arm runs): it holds the share on the last block's direction well under bare's at every read; its bits a byte at 4,960 within max(0.001, the two bare seeds' spread) of bare; its exit H inside bare's spread. A form that costs more than that bar is normalizing, not bounding.

**Length:** about 27 minutes a Castor arm and 44 a Pollux-layout arm on an RTX PRO 6000: about 4.9 hours for all nine; the cell can be stopped and rerun (each arm resumes from the repo, finished arms are read back).

**Token:** `HF_TOKEN` with write access to AbstractPhil/alephllm-nano-beatrix-training (the records land under `v1_lastfix/<arm>/`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.28"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import lastfix_screen as LS
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 28), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/nano_v1_lastfix'                      # the session folder: the starts, the runs, the records
REPO = LS.REPO                                        # AbstractPhil/alephllm-nano-beatrix-training, under v1_lastfix/<arm>/
NAMES = [a['name'] for a in LS.ARMS]                  # the nine arms in order (the deciding Castor arms first)
MAX_HOURS = None                                      # this session's budget in hours (None: until every arm is done)
print('arms:', ', '.join(NAMES), '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: every arm's start (checked against a fresh build), the length, the push probe. Trains nothing.
import json, time
births = {n: LS.write_arm_birth(LS.BY_NAME[n], f'{OUT}/births') for n in NAMES}
print('starts:', ', '.join(f"{n} <- {LS.BY_NAME[n]['craft']} seed {LS.BY_NAME[n]['seed']} {LS.BY_NAME[n]['mode'] or 'bare'}" for n in NAMES))
step_s = {'nano-castor-v1': 0.299, 'nano-pollux-dual-canon-v1': 0.493}      # the v1 bench on an RTX PRO 6000 at 16 x 4 (2026-10-14)
mins = {n: (4960 * step_s[LS.BY_NAME[n]['craft']] + 12 * 25) / 60 for n in NAMES}
print(f'-> about {sum(mins.values()) / 60:.1f} h on an RTX PRO 6000 (training plus a full eval and the gauge every 500 steps)')
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'{LS.PREFIX}/preflight/push_probe.json'
    api.upload_file(path_or_fileobj=json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                                'card': torch.cuda.get_device_name(0), 'arms': NAMES}).encode(),
                    path_in_repo=path, repo_id=REPO, commit_message='v1 last-write screen: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE SCREEN: arm by arm; the trainer's own lines and the screen's gauge lines ([lastfix]) print as they come. Rerun the cell to
# continue (a new session too): each arm resumes from the repo, finished arms are read back. The stop button saves the running arm first.
RECS = LS.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, max_hours=MAX_HOURS)
from IPython.display import Markdown, display
display(Markdown('**At the end of the warmup (step 1,145)**\n\n' + LS.table(RECS, 1145)))
display(Markdown('**At the end (each arm\'s last read)**\n\n' + LS.table(RECS)))

In [ ]:
# C4 — THE CURVES AND THE TABLES from the repo (every session; tokenless: this session's files). Trains nothing.
import json
from IPython.display import Markdown, display
recs, logdir = {}, f'{OUT}/runs'
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                            allow_patterns=[f'{LS.PREFIX}/{n}/runs/*' for n in NAMES] + [f'{LS.PREFIX}/{n}/screen_record.json' for n in NAMES])
    logdir = f'{hub}/{LS.PREFIX}'
    for n in NAMES:
        p = f'{hub}/{LS.PREFIX}/{n}/screen_record.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
else:
    for n in NAMES:
        p = f'{OUT}/lastfix/{n}.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
display(Markdown('**At the end of the warmup (step 1,145)**\n\n' + LS.table(recs, 1145)))
display(Markdown('**Each arm\'s last read**\n\n' + LS.table(recs)))
%load_ext tensorboard
%tensorboard --logdir {logdir}

### Reading the result

| read | bare (the craft as the v1 train ran it) | a form that works |
|---|---|---|
| share on the last block's direction | climbs toward 0.9+ | stays well under bare at every read |
| the last bank's raw write (before the bound) | grows tens of times | may still grow: the bound acts after it, so the share above is the read that counts |
| the last block's rank | falls to ~13 of 256 | stays higher |
| bits a byte at 4,960 | the reference | within max(0.001, the bare seeds' spread) |
| the exit's H on random bytes | the reference | inside bare's spread (the drift kept) |

The form that passes goes into the v1 crafts (a config flag, one line in the v1 notebook), and the v1 train starts from scratch.

### Changing the screen
| to | the edit in C1 |
|---|---|
| only the Castor arms | `NAMES = [n for n in NAMES if n.startswith('c-')]` |
| a session budget | `MAX_HOURS = 2` |